In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [3]:
rc_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\Assam\assam_rc_2024-11_reduced.json")
tenders_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\Flood Tenders Dashboard\Assam 2017-2024 #1\Assam 2017-2024 V2\floodtenders_RCgeotagged_2017-2025March.csv')

In [4]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [5]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in rc_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [6]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
rc_gdf['geometry'] = rc_gdf['geometry'].apply(simplify_multipolygon)

geo_fixed = gpd.GeoDataFrame(rc_gdf, geometry='geometry')


In [7]:
rc_gdf

,revenue_ci,revenue_cr,HQ,are_new,dtname,object_id,dtcode11,geometry
0,Gossaigaon (Pt),Gossaigaon (Pt),None,1069,KOKRAJHAR,18-300-00101,18-300,"POLYGON ((90.18542 26.64048, 90.19629 26.67040..."
1,Bhowraguri,Bhawraguri,None,159,KOKRAJHAR,18-300-00102,18-300,"POLYGON ((90.10500 26.33134, 90.11650 26.35670..."
2,Dotoma,Dotoma,None,304,KOKRAJHAR,18-300-00103,18-300,"POLYGON ((90.20515 26.37123, 90.18616 26.40796..."
3,Kokrajhar (Pt),Kokrajhar (Pt),y,990,KOKRAJHAR,18-300-00104,18-300,"POLYGON ((90.36157 26.59586, 90.34777 26.63903..."
4,Bagribari (Pt),Bagribari (Pt),None,281,KOKRAJHAR,18-300-00105,18-300,"POLYGON ((90.10500 26.33134, 90.05091 26.29692..."
...,...,...,...,...,...,...,...,...
175,Sapekhati,Sapekhati,None,394,CHARAIDEO,18-755-00278,18-755,"POLYGON ((95.10936 27.12364, 95.10667 27.11599..."
176,Sonari,Sonari,y,385,CHARAIDEO,18-755-00279,18-755,"POLYGON ((94.95742 27.06354, 94.96116 27.05681..."
177,Ujani Majuli,Ujani Majuli,None,322,MAJULI,18-760-00280,18-760,"POLYGON ((94.22459 27.03514, 94.22794 27.00592..."
178,Majuli,Majuli,None,648,MAJULI,18-760-00281,18-760,"POLYGON ((94.56755 27.17914, 94.55904 27.17835..."


In [8]:
merged_gdf = tenders_df.merge(geo_fixed[['geometry','revenue_ci','dtname']], right_on=['revenue_ci','dtname'], left_on=['REVENUE_CIRCLE_FINALISED','DISTRICT_FINALISED'], how='left')
merged_gdf = merged_gdf.drop(columns=['revenue_ci','dtname','tender_revenueci_location'])
merged_gdf

,Unnamed: 0,Tender ID,tender_externalreference,tender_title,Work Description,Tender Category,Tender Type,Form of contract,Product Category,Is Multi Currency Allowed For BOQ,...,tender_district_location,BTC_flag,DISTRICT_FINALISED,tender_villages,tender_block,tender_subdistrict,tender_revenueci,HQ_flag,REVENUE_CIRCLE_FINALISED,geometry
0,91,2017_DoWR_2083_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal 1,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,False,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250..."
1,92,2017_DoWR_2238_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal Drainage Basin Ph-II,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,False,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250..."
2,126,2018_DoWR_5796_1,HAILAKANDI/SDRF/2017-18/1,IM at Kalinagar Pk-1,Immediate measures to dyke along l/b of river ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,False,HAILAKANDI,"'MOHANPUR', 'KALINAGAR'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250..."
3,288,2019_DoWR_11496_1,HAILAKANDI/2018-19/SDRF/II,IM at Matijuri,Immediate measures to Restoration for damages ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,False,HAILAKANDI,"'MATIJURI', 'Matijuri'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250..."
4,290,2019_DoWR_11502_1,HAILAKANDI/2018-19/SDRF/VII,IM at Rupacherra Ghat,Immediate measures to dyke along L/B of river ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,False,HAILAKANDI,'SAHABAD',HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1148,1040,2024_DoWR_36038_1,MAJULI/2023-24/NIDA/I,Protection of Pekamoni and its adjoining areas...,Protection of Pekamoni and its adjoining areas...,Works,Open Tender,Works,Civil Works,No,...,MAJULI,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
1149,1094,2024_ICD_38053_1,505342/173 dated 09.07.2024,"Construction of Boundary Wall, Land Developmen...","Construction of Boundary Wall, Land Developmen...",Works,Open Tender,Item Rate,Civil Works,No,...,GOALPARA,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
1150,1096,2024_PWD_39765_1,SE/MRC/TB/NIT/01/2023-24/17,25 Package No. MRC/TAC/2024-25/02(Gr-X),1.Construction of road with RCC culvert from L...,Works,Open Tender,Item Rate,Civil Works – Roads,No,...,MORIGAON,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
1151,1115,2024_DoWR_38055_1,NAGAON/2023-24/NIDA/III,Construction of embankment on the right bank o...,Construction of embankment on the right bank o...,Works,Open Tender,Works,Civil Works,No,...,NALBARI,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None


In [9]:
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)
merged_gdf

,Unnamed: 0,Tender ID,tender_externalreference,tender_title,Work Description,Tender Category,Tender Type,Form of contract,Product Category,Is Multi Currency Allowed For BOQ,...,BTC_flag,DISTRICT_FINALISED,tender_villages,tender_block,tender_subdistrict,tender_revenueci,HQ_flag,REVENUE_CIRCLE_FINALISED,geometry,polygons
0,91,2017_DoWR_2083_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal 1,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,False,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5..."
1,92,2017_DoWR_2238_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal Drainage Basin Ph-II,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,False,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5..."
2,126,2018_DoWR_5796_1,HAILAKANDI/SDRF/2017-18/1,IM at Kalinagar Pk-1,Immediate measures to dyke along l/b of river ...,Works,Open Tender,Works,Civil Works,No,...,False,HAILAKANDI,"'MOHANPUR', 'KALINAGAR'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5..."
3,288,2019_DoWR_11496_1,HAILAKANDI/2018-19/SDRF/II,IM at Matijuri,Immediate measures to Restoration for damages ...,Works,Open Tender,Works,Civil Works,No,...,False,HAILAKANDI,"'MATIJURI', 'Matijuri'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5..."
4,290,2019_DoWR_11502_1,HAILAKANDI/2018-19/SDRF/VII,IM at Rupacherra Ghat,Immediate measures to dyke along L/B of river ...,Works,Open Tender,Works,Civil Works,No,...,False,HAILAKANDI,'SAHABAD',HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1148,1040,2024_DoWR_36038_1,MAJULI/2023-24/NIDA/I,Protection of Pekamoni and its adjoining areas...,Protection of Pekamoni and its adjoining areas...,Works,Open Tender,Works,Civil Works,No,...,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
1149,1094,2024_ICD_38053_1,505342/173 dated 09.07.2024,"Construction of Boundary Wall, Land Developmen...","Construction of Boundary Wall, Land Developmen...",Works,Open Tender,Item Rate,Civil Works,No,...,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
1150,1096,2024_PWD_39765_1,SE/MRC/TB/NIT/01/2023-24/17,25 Package No. MRC/TAC/2024-25/02(Gr-X),1.Construction of road with RCC culvert from L...,Works,Open Tender,Item Rate,Civil Works – Roads,No,...,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
1151,1115,2024_DoWR_38055_1,NAGAON/2023-24/NIDA/III,Construction of embankment on the right bank o...,Construction of embankment on the right bank o...,Works,Open Tender,Works,Civil Works,No,...,False,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None


In [10]:
merged_gdf = merged_gdf.dropna(subset =['object_id'])

KeyError: ['object_id']

In [11]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged_gdf['timeperiod'] = merged_gdf['month'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged_gdf['timeperiod'] = pd.to_datetime(merged_gdf['timeperiod'], format='%Y-%m')


merged_gdf['timeperiod'] = merged_gdf['timeperiod'].dt.strftime('%Y-%m')

In [12]:
merged_gdf.columns = merged_gdf.columns.str.lower().str.replace(' ', '_')
merged_gdf = merged_gdf.rename(columns={'contract_date_:':'contract_date','bid_validity(days)':'bid_validity_days','tender_value_in_₹':'tender_value_in_rupees'})
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].str.replace(',', '')
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].astype(float)
merged_gdf = merged_gdf.dropna(subset=['awarded_value', 'tender_value_in_rupees','district_finalised'])
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].str.replace(',', '')
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].astype(float)
merged_gdf

,unnamed:_0,tender_id,tender_externalreference,tender_title,work_description,tender_category,tender_type,form_of_contract,product_category,is_multi_currency_allowed_for_boq,...,district_finalised,tender_villages,tender_block,tender_subdistrict,tender_revenueci,hq_flag,revenue_circle_finalised,geometry,polygons,timeperiod
0,91,2017_DoWR_2083_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal 1,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
1,92,2017_DoWR_2238_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal Drainage Basin Ph-II,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
2,126,2018_DoWR_5796_1,HAILAKANDI/SDRF/2017-18/1,IM at Kalinagar Pk-1,Immediate measures to dyke along l/b of river ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MOHANPUR', 'KALINAGAR'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
3,288,2019_DoWR_11496_1,HAILAKANDI/2018-19/SDRF/II,IM at Matijuri,Immediate measures to Restoration for damages ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MATIJURI', 'Matijuri'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
5,295,2019_DoWR_11622_1,HAILAKANDI/2018-19/SDRF/V,IM to protect Nandigram Area,Immediate measures to protect Nandigram area f...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1148,1040,2024_DoWR_36038_1,MAJULI/2023-24/NIDA/I,Protection of Pekamoni and its adjoining areas...,Protection of Pekamoni and its adjoining areas...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-07
1149,1094,2024_ICD_38053_1,505342/173 dated 09.07.2024,"Construction of Boundary Wall, Land Developmen...","Construction of Boundary Wall, Land Developmen...",Works,Open Tender,Item Rate,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-12
1150,1096,2024_PWD_39765_1,SE/MRC/TB/NIT/01/2023-24/17,25 Package No. MRC/TAC/2024-25/02(Gr-X),1.Construction of road with RCC culvert from L...,Works,Open Tender,Item Rate,Civil Works – Roads,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-12
1151,1115,2024_DoWR_38055_1,NAGAON/2023-24/NIDA/III,Construction of embankment on the right bank o...,Construction of embankment on the right bank o...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2025-02


In [13]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\Flood Tenders Dashboard\Assam 2017-2024 V2\assam_tenders.csv')